# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/Andersen_NCBI_Virus_GISAID/" 

os.chdir(downloads)

## Collect user input

In [3]:
# locations = input("Locations (separate with commas and no spaces): ")
# start_date = input("Start date (format: YYYY-MM-DD): ")
# end_date = input("End date (format: YYYY-MM-DD): ")

In [ ]:
locations = "Antarctica,North America,South America"
start_date = "2021-11-01"
end_date = "2025-08-01"

## Create directories if needed

In [5]:
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "Combinations/Andersen_NCBI_Virus/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

print(complete_files)

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/


## Get list of genotypes and states

In [ ]:
# Get list of genotypes and states

# os.chdir("C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/")
os.chdir(references)

states = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

# genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]

genotypes = ["B3.13", "D1.1", "D1.3"] # , "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

## Download all files, run through all files, convert fasta files to dataframes, and separate them into different dataframes based on segment ##

In [7]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it
    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Have user type in username and password
            username = input("Username: ")
            password = input("Password: ")
            browser = input("Browser: ")
            sleep_time = input("Seconds to sleep in between clicks: ")

            open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 


for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # file_name = "_".join(file_name.split(" "))
        
        os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))

for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        # if ".csv" in file_name:
        #     metadata = pd.read_csv(file_name)
        #     all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            # print(fasta_file[fasta_file["Geo_Location"] != "USA"])
            # break 
            all_fasta_files.append(fasta_file)
    break 

print(len(all_metadata_files))
print(len(all_fasta_files))

# print(all_metadata_files)

# all_metadata_files = [all_metadata_files[0]]
# all_fasta_files = [all_fasta_files[1]]

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2025-07-25_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2025-07-25_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta
1
1


In [8]:
# Get metadata

def separate_fasta_by_segs(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first
    # Dummy host type -- we'll actually add this in later
    # b313_fasta["Host_Type"] = "other"
    # d11_fasta["Host_Type"] = "other"

    unique_segments = list(set(fasta["Segment"])) # Get list of segments
    # genotypes = ["B3.13", "D1.1"]
    # genotype_fastas = {"B3.13": b313_fasta, "D1.1": d11_fasta}

    # “>EPI_ID/Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for fasta_gen in genotypes: # .keys(): # For each genotype
        print(fasta_gen)
        for seg in unique_segments: # For each segment

            xls = metadata[metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == fasta_gen] # Get only the metadata corresponding to that genotype

            # print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Isolate_Id"])

            # print(d11_xls)

            # FASTA
            # if "Identifier" in fasta.columns:
            #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
            # else:           
            #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])

            fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            print(fasta_seg_pre)

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = fasta_gen

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name"] + "|" + fasta_seg["Subtype"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] + "\n"
            fasta_seg["New_Name"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            # print(fasta_seg)

    return segment_fastas, unique_segments


# Separate fastas by segment
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):
    # print(all_fasta_files)
    # print(fasta)
    # print(i)
    metadata = all_metadata_files[i]
    # print(metadata)
    # print(fasta.loc[i, "Isolate_Name"])
    
    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segs(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment
    # print(fastas[0])
    segment_fastas.append(fastas)

# print(segment_fastas[0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

B3.2
                                                   Header     Isolate_Id  \
1720    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1721    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1722    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1723    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1724    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
...                                                   ...            ...   
146651  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146652  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146653  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146654  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146655  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   

                                       Isolate_Name Subtype Segment Location  \
17

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name


                                                   Header     Isolate_Id  \
1720    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1721    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1722    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1723    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1724    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
...                                                   ...            ...   
146651  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146652  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146653  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146654  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146655  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   

                                       Isolate_Name Subtype Segment Location  \
1720   

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name


                                                   Header     Isolate_Id  \
1720    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1721    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1722    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1723    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
1724    EPI_ISL_19660845|A/turkey_vulture/Wyoming/23-0...  23-012565-001   
...                                                   ...            ...   
146651  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146652  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146653  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146654  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   
146655  EPI_ISL_16023345|A/Red_Fox/AB/FAV-0835-01/2022...    FAV-0835-01   

                                       Isolate_Name Subtype Segment Location  \
1720   

...

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen


                                                   Header  Isolate_Id  \
440     EPI_ISL_19792238|A/Chicken/NS/FAV-0076-2/2025|...  FAV-0076-2   
441     EPI_ISL_19792238|A/Chicken/NS/FAV-0076-2/2025|...  FAV-0076-2   
442     EPI_ISL_19792238|A/Chicken/NS/FAV-0076-2/2025|...  FAV-0076-2   
443     EPI_ISL_19792238|A/Chicken/NS/FAV-0076-2/2025|...  FAV-0076-2   
444     EPI_ISL_19792238|A/Chicken/NS/FAV-0076-2/2025|...  FAV-0076-2   
...                                                   ...         ...   
146475  EPI_ISL_19726293|A/Nevada/10/2025|A_/_H5N1|PA|...          10   
146476  EPI_ISL_19726293|A/Nevada/10/2025|A_/_H5N1|PB2...          10   
146477  EPI_ISL_19726293|A/Nevada/10/2025|A_/_H5N1|PB1...          10   
146478  EPI_ISL_19726293|A/Nevada/10/2025|A_/_H5N1|NA|...          10   
146479  EPI_ISL_19726293|A/Nevada/10/2025|A_/_H5N1|HA|...          10   

                        Isolate_Name Subtype Segment Location Geo_Location  \
440     A/Chicken/NS/FAV-0076-2/2025    H5N1 

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

D1.3
                                                   Header    Isolate_Id  \
1232    EPI_ISL_19792283|A/cooper's_hawk/USA/003149-00...  003149-003-v   
1233    EPI_ISL_19792283|A/cooper's_hawk/USA/003149-00...  003149-003-v   
1234    EPI_ISL_19792283|A/cooper's_hawk/USA/003149-00...  003149-003-v   
1235    EPI_ISL_19792283|A/cooper's_hawk/USA/003149-00...  003149-003-v   
1236    EPI_ISL_19792283|A/cooper's_hawk/USA/003149-00...  003149-003-v   
...                                                   ...           ...   
141355  EPI_ISL_19756112|A/turkey/Ohio/004606-001/2025...    004606-001   
141356  EPI_ISL_19756112|A/turkey/Ohio/004606-001/2025...    004606-001   
141357  EPI_ISL_19756112|A/turkey/Ohio/004606-001/2025...    004606-001   
141358  EPI_ISL_19756112|A/turkey/Ohio/004606-001/2025...    004606-001   
141359  EPI_ISL_19756112|A/turkey/Ohio/004606-001/2025...    004606-001   

                                 Isolate_Name Subtype Segment Location  \
1232    A/cooper's_h

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_40700\37931878.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

## De-Duplication

In [9]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {}
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
        fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
        andersen_ncbi[segment_genotype] = fasta_file
    

In [10]:
# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

ha_only = [] # Only do one segment, as the others are identical 
for gisaid_fasta in segment_fastas:
    for genotype_gisaid_fasta in gisaid_fasta:
        # print(genotype_gisaid_fasta)
        if "HA" in genotype_gisaid_fasta["Segment"].values:
            genotype_gisaid_fasta["Partials"] = genotype_gisaid_fasta["Isolate_Id"].apply(partial_isolate)
            genotype_gisaid_fasta["Year"] = genotype_gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
            genotype_gisaid_fasta = genotype_gisaid_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
            ha_only.append(genotype_gisaid_fasta)
        # print(genotype_gisaid_fasta)
        
# print(ha_only)
    
andersen_ncbi_genotypes = {}
for key in andersen_ncbi:
    andersen_ncbi_fasta = andersen_ncbi[key]
    # Find partial Isolate IDs
    andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    # print("Andersen:", andersen_ncbi_fasta["Partials"])
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year))
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    # print(andersen_ncbi_fasta)
    if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys() and andersen_ncbi_fasta["Segment"].values[0] == "HA": # If we haven't already seen this genotype, and if HA
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype dictionary
        # print(andersen_ncbi_fasta)
    # break 

# print(andersen_ncbi_genotypes)

gisaid_only_dfs = {} # We need these so that we know what sequences to throw out in the other segments too
for gisaid_genotype in ha_only: # Each dataframe is unique in genotype
    # print(gisaid_genotype[gisaid_genotype["Host_Type"] == "human"])
    genotype = gisaid_genotype["Genotype"].values[0]
    andersen_ncbi_fasta = pd.DataFrame()
    if genotype in andersen_ncbi_genotypes.keys(): # and genotype not in gisaid_only_dfs.keys(): # If it's in Andersen and we haven't seen it before here
        andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
        print(len(andersen_ncbi_fasta))
        print(len(gisaid_genotype))
        deduplicated = pd.concat([gisaid_genotype,andersen_ncbi_fasta]).drop_duplicates(subset=["Partials", "Year"], keep="last") 
        print(len(deduplicated))
        gisaid_only_dfs[genotype] = deduplicated
    # else:
    #     deduplicated = gisaid_genotype
    
# print(gisaid_only_dfs["D1.3"][gisaid_only_dfs["D1.3"]["Host_Type"] == "cattle"])
    
        # print(deduplicated[deduplicated["Host_Type"] == "human"])

# print(gisaid_only_dfs)

# Identify sequences we are keeping
genotype_seq_keep = {}
for genotype in gisaid_only_dfs:
    deduplicated = gisaid_only_dfs[genotype]
    genotype_seq_keep[genotype] = list(deduplicated["Identifier"])

# Keep in other segments only the sequences we kept in HA
kept_seqs = []
for genotype_group in segment_fastas:
    # print(genotype_group)
    for gisaid_df in genotype_group:
        # print(gisaid_df)
        if len(gisaid_df["Genotype"]) > 0: # If there are sequences
            genotype = gisaid_df["Genotype"].values[0]
            print(len(genotype_seq_keep[genotype]))
            gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]
            print(len(gisaid_df_new))
            kept_seqs.append(gisaid_df_new)

print(len(kept_seqs))

27
1891
1896
68
952
955
6
122
122
2
158
158
101
301
330
4601
4018
4893
2898
2372
3143
313
105
323
1896
1869
1896
1869
1896
1869
1896
1869
1896
1869
1896
1869
1896
1869
1896
1869
955
887
955
887
955
887
955
887
955
887
955
887
955
887
955
887
122
116
122
116
122
116
122
116
122
116
122
116
122
116
122
116
158
156
158
156
158
156
158
156
158
156
158
156
158
156
158
156
330
229
330
229
330
229
330
229
330
229
330
229
330
229
330
229
4893
292
4893
292
4893
292
4893
292
4893
292
4893
292
4893
292
4893
292
3143
245
3143
245
3143
245
3143
245
3143
245
3143
245
3143
245
3143
245
323
10
323
10
323
10
323
10
323
10
323
10
323
10
323
10
64


Create animal reference if needed 

In [11]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(home)
animals_df.to_csv("animals_ref_to_sort.csv")

['great_black_backed_gull', 'northern_gannet', 'bar-headed_goose', 'texas', 'feral_cat', 'sterna_hirundo', 'white-winged_scoter', 'willet', 'domestic_goose', 'gyrfalcon', 'common_loon', 'glaucous_gull', 'gannet', 'waterfowl', 'dove', 'savannah_cat', 'comon_goldeneye', 'otaria_flavescens', 'gallus_gallus', 'khaki_campbell_duck', 'wild_goose', 'european_starling', 'american_wood_stork', 'swallow', 'cougar', 'broad_winged_hawk', 'greater_white-fronted_goose', 'dog', 'numida_meleagris', 'rock_pigeon', 'great_black-backed_gull', 'bonapartes_gull', 'american_green-winged_teal', 'procellaria_aequinoctialis', 'south_american_tern', 'iceland_gull', 'sanderling', 'wild_bird', 'lion', 'falcon', 'feline', "cooper'ss_hawk", 'american_goshawk', 'chinese_ringneck_pheasant', 'house_fly', 'sandwich_tern', 'flamingo', 'cormorant', 'peafowl', 'backyard_turkey', 'mallard_duck', 'brandt_goose', 'black_crowned_night-heron', 'black_bear', 'black_swan', 'gallus', 'king_vulture', 'thayers_gull', 'southern_elep

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [12]:
# 16 files needed
huge_fasta = pd.DataFrame()

for fastas in kept_seqs: # 7 batches
    # print(fastas.columns)
    # print(len(fastas))
    # break
    # for f in fastas: # 16 files per batch 
        # print(f)
        # break 
    huge_fasta = pd.concat([huge_fasta, fastas])

print(huge_fasta.columns)

# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

# genotypes = ["B3.13", "D1.1"] # , "D1.3"]
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)

print(big_fasta)

# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    fasta_df = fasta[["New_Name", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_GISAID_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            output_file.write(item)
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

print(len(huge_fasta))
print(len(big_fastas[0]))

Index(['Header', 'Isolate_Id', 'Isolate_Name', 'Subtype', 'Segment',
       'Location', 'Geo_Location', 'Date Collected', 'Species', 'Sequence',
       'Identifier', 'Host_Type', 'Genotype', 'New_Name', 'Partials', 'Year'],
      dtype='object')
                                                   Header   Isolate_Id  \
5375    EPI_ISL_19793724|A/Ostrich/BC/FAV-0003-02/2025...  FAV-0003-02   
31117   EPI_ISL_19737077|A/chicken/USA/000491-004/2025...   000491-004   
38749   EPI_ISL_19737333|A/turkey/Ohio/001329-003/2024...   001329-003   
38797   EPI_ISL_19737334|A/turkey/Ohio/001329-002/2024...   001329-002   
38829   EPI_ISL_19737328|A/turkey/Ohio/001339-001/2024...   001339-001   
...                                                   ...          ...   
38840   EPI_ISL_19737331|A/turkey/Ohio/001338-001/2024...   001338-001   
39224   EPI_ISL_19737310|A/turkey/Ohio/001329-004/2024...   001329-004   
39320   EPI_ISL_19737306|A/turkey/Ohio/001337-002/2024...   001337-002   
39440   EPI_IS

## Concatenate to Andersen_NCBI files and save

In [13]:
# Concat
os.chdir(andersen_ncbi_virus_gisaid)

for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        for dirpath1, dirs1, files1 in os.walk(gisaid_files):
            for file1 in files1:
                file_name1 = os.path.join(dirpath1, file1)
                # print(file_name1)
                
                if "_".join(file_name.split("/")[-1].split("_")[0:2]) == "_".join(file_name1.split("/")[-1].split("_")[0:2]): # If they match
                    print("_".join(file_name.split("/")[-1].split("_")[0:2]))
                    output_path = complete_files + "all_" + file_name.split("/")[-1] # Genotype and Segment should all be the same
                    output_file = open(output_path, "w")
                    with open(file_name) as f:
                        for line in f.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                        f.close()
                    with open(file_name1) as f1:
                       for line in f1.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                    f1.close()  

                    output_file.close()
                

            break 
    break 

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.5_HA
B3.5_MP
B3.5_NA
B3.5_NP
B3.5_NS
B3.5_PA
B3.5_PB1
B3.5_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
B3.7_HA
B3.7_MP
B3.7_NA
B3.7_NP
B3.7_NS
B3.7_PA
B3.7_PB1
B3.7_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
